We want to combine the weather data with the delay data because doing it in the notebook every time takes me about 5 minutes per execution.

In [4]:
import pandas as pd

In [5]:
weather=pd.read_csv('../data/weatherstats_toronto_daily.csv')
#drop columns that consist exclusively of NaNs
weather.dropna(axis=1, how='all', inplace=True)
#Convert date from string to datetime type
weather.date = pd.to_datetime(weather.date)

#fill remainder of the NaNs with '0'
weather.fillna(0, inplace=True)

columns = weather.columns
#the following columns have a deterministic relationship with the temperature so we are dropping them
columns = [x for x in columns if 'degdays' not in x]

# Choosing avg stats vs max/min stats
columns = [x for x in columns if 'max' not in x and 'min' not in x]
# Removing 'hourly data'
columns = [x for x in columns if 'hourly' not in x]
# Picking one time format
columns = [x for x in columns if 'sunrise_f' != x and 'sunset_f' != x]
columns = [x for x in columns if 'unixtime' not in x]
# Picking station pressure vs sea pressure
columns = [x for x in columns if 'avg_pressure_sea' != x]
# Drop 'wind_gust_dir_10s' and 'snow_on_ground' because of lack of data
columns = [x for x in columns if x not in ['wind_gust_dir_10s', 'snow_on_ground']]

weather = weather[columns]

display(weather.head())

,date,avg_temperature,avg_relative_humidity,avg_dew_point,avg_wind_speed,avg_pressure_station,avg_visibility,avg_health_index,precipitation,rain,snow,sunrise_hhmm,sunset_hhmm,daylight,avg_cloud_cover_8
0,2026-02-21,0.40,87.0,-0.9,10.5,99.13,14450,3.0,2.0,1.1,0.4,07:08:00,17:56:00,10.80,7.5
1,2026-02-20,1.80,88.0,0.2,21.0,98.07,14450,2.8,5.2,5.2,0.0,07:09:00,17:54:00,10.75,7.5
2,2026-02-19,0.29,76.5,-3.6,15.0,99.12,21700,3.1,0.0,0.0,0.0,07:11:00,17:53:00,10.70,7.5
3,2026-02-18,0.29,77.5,-3.5,17.5,99.06,12650,4.0,15.1,6.7,6.8,07:12:00,17:52:00,10.67,6.5
4,2026-02-17,3.90,80.5,0.4,9.0,99.52,6450,4.8,0.0,0.0,0.0,07:14:00,17:50:00,10.60,4.0


In [6]:
'''
We add the weather data as columns into our delay data.
'''
delay_data = pd.read_csv('../data/2014-2024data.csv')

for parameter in weather.columns:
    data = []
    for k in range(delay_data.shape[0]):
        current_date = delay_data['Date'].iloc[k]
        data.append(weather[weather.date == current_date][parameter].iloc[0])
    delay_data[parameter] = data

display(delay_data.head())

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_46819/4218979869.py:4: DtypeWarning: Columns (0: Line) have mixed types. Specify dtype option on import or set low_memory=False.
  delay_data = pd.read_csv('../data/2014-2024data.csv')


,Time,Day,Location,Incident,Min Delay,Min Gap,Vehicle,Date,Line,Bound,...,avg_pressure_station,avg_visibility,avg_health_index,precipitation,rain,snow,sunrise_hhmm,sunset_hhmm,daylight,avg_cloud_cover_8
0,06:31:00,Thursday,Dundas and Roncesvalles,Late Leaving Garage,4.0,8.0,4018.0,2014-01-02,505.0,E/B,...,100.19,13650,2.5,0.8,0.0,1.0,07:52:00,16:52:00,9.0,7.0
1,12:43:00,Thursday,King and Shaw,Utilized Off Route,20.0,22.0,4128.0,2014-01-02,504.0,E/B,...,100.19,13650,2.5,0.8,0.0,1.0,07:52:00,16:52:00,9.0,7.0
2,14:01:00,Thursday,Kingston road and Bingham,Held By,13.0,19.0,4016.0,2014-01-02,501.0,W/B,...,100.19,13650,2.5,0.8,0.0,1.0,07:52:00,16:52:00,9.0,7.0
3,14:22:00,Thursday,King St. and Roncesvalles Ave.,Investigation,7.0,11.0,4175.0,2014-01-02,504.0,W/B,...,100.19,13650,2.5,0.8,0.0,1.0,07:52:00,16:52:00,9.0,7.0
4,16:42:00,Thursday,King and Bathurst,Utilized Off Route,3.0,6.0,4080.0,2014-01-02,504.0,E/B,...,100.19,13650,2.5,0.8,0.0,1.0,07:52:00,16:52:00,9.0,7.0


In [7]:
# It turns out sunrise/sunset are not timestamped, so let's fix that.
import datetime
try:
    delay_data['sunrise_hhmm'] = delay_data['sunrise_hhmm'].apply(lambda x: datetime.time(hour=int(x[:2]), minute=int(x[3:5]), second=int(x[6:8])))
    delay_data['sunset_hhmm'] = delay_data['sunset_hhmm'].apply(lambda x: datetime.time(hour=int(x[:2]), minute=int(x[3:5]), second=int(x[6:8])))
except:
    print('Warning: Did you already run this cell?')

In [11]:
delay_data.to_csv('../data/2014-2024data_with_weather.csv', index=False)